## Implement Databricks Subagents with LangChain

### Installing Utilities and Libraries

In [ ]:
%pip install databricks-langchain==0.12.1 langchain-community==0.4.1 langchain-experimental==0.4.1 databricks-mcp==0.9.2 mcp==2.0.0

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Instantiate the ChatDatabricks Class

In [ ]:
import json
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint="databricks-claude-sonnet-4-5",
    temperature=0.1,
    max_tokens=250,
)

### Load the Web Search MCP Tool

In [ ]:
from databricks.sdk import WorkspaceClient
from langchain_mcp_adapters.client import MultiServerMCPClient

w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")

workspace_host = w.config.host.rstrip("/")

web_search_mcp_url = (
    "ENTER_YOUR_MCP_SERVER_URL_HERE"
)

mcp_client = MultiServerMCPClient(
    {
        "web_search": {
            "transport": "streamable_http",
            "url": web_search_mcp_url,
            "headers": {
                "Authorization": f"Bearer {token}"
            },
        }
    }
)

mcp_tools = await mcp_client.get_tools()

for tool in mcp_tools:
    print(tool.name)

### Create the Researcher Subagent

In [ ]:
from langchain.agents import create_agent

research_model = model.bind_tools([search_tool])

research_agent = create_agent(
    model=research_model,
    tools=[],
    system_prompt="""
You are a senior market research analyst.

Use web search whenever current information,
competitor research, or market trends are required.

Always cite your findings.
"""
)

### Create the Content Writer Subagent

In [ ]:
content_writer = create_agent(
    model=model,
    tools=[],
    system_prompt="""
You are an expert Marketing Content Writer.

Create:

- LinkedIn posts
- Product launch announcements
- Marketing copy
- Promotional content

Write in a professional and engaging tone.
"""
)

### Wrap the Subagents as tools

In [ ]:
from langchain.tools import tool

@tool
def research(query: str) -> str:
    """
    Research a topic and return findings.
    """

    print("\n" + "=" * 60)
    print("Executing Research Agent")
    print("=" * 60)
    print(query)
    print()

    result = research_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    answer = result["messages"][-1].text

    print("\nResearch Agent Finished.\n")

    return answer


@tool
def write_marketing_copy(prompt: str) -> str:
    """
    Create marketing content.
    """

    print("\n" + "=" * 60)
    print("Executing Content Writer Agent")
    print("=" * 60)
    print(prompt)
    print()

    result = content_writer.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    answer = result["messages"][-1].text

    print("\nContent Writer Finished.\n")

    return answer

### Create the Supervisor Agent

In [ ]:
supervisor = create_agent(
    model=model,

    tools=[
        research,
        write_marketing_copy
    ],

    system_prompt="""
You are the Marketing Supervisor.

Delegate work to the appropriate specialist.

Use:

- research()
    For customer analysis, competitors, personas, positioning.

- write_marketing_copy()
    For LinkedIn posts, launch announcements and promotional content.

Combine the results into one final response.
"""
)

### Invoke the Supervisor Agent

In [ ]:
response = supervisor.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
We are launching a new AI-powered fitness smartwatch called FitSense AI.

First identify:

- Target audience
- Customer pain points
- Current fitness wearable trends

Then create a professional LinkedIn launch announcement.
"""
            }
        ]
    }
)

print("\n" + "=" * 60)
print("FINAL RESPONSE")
print("=" * 60)
print(response["messages"][-1].text)